<a href="https://colab.research.google.com/github/sergioGarcia91/SeismicUP/blob/main/03_Analisis_de_subcatalogos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

En este notebook se procederá con el análisis del catálogo sísmico dividiéndolo en tres subcatalogos:

1. `FC` (Full Catalog): corresponde a todos los eventos descargados.
2. `BSN` (Bucaramanga Seismic Nest Catalog): corresponde a los eventos asociados al área de estudio de García-Arias et al. (2026).
3. `CC` (Campo Colorado Catalog): corresponde a los eventos contenidos en un área de 100x100 km, centrada en el punto de referencia del Campo Colorado (X = 4'919.222,973; Y = 2'308.553,476; EPSG:9377).

A continuación se presentan los límites espaciales de cada catálogo expresados en coordenadas geográficas EPSG:4326:

| Catálogo | Longitud (min, max) | Latitud (min, max) |
| :------: | :-----------------: | :----------------: |
|    FC    |    -76.00, -71.00   |     3.00, 11.00    |
|    BSN   |    -73.40, -72.80   |     6.50, 7.10     |
|    CC    |    -74.19, -73.26   |     6.32, 7.25     |






# Inicio

In [ ]:
!git clone https://github.com/sergioGarcia91/SeismicUP.git

In [ ]:
# para incluir la libreria clonada
import sys
sys.path.append("/content/SeismicUP")

import seismicup as sup

In [ ]:
!pip -q install python-calamine

In [ ]:
!pip3 install contextily

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import os
import re
import contextily as cx #para el basemap en geopandas
import xyzservices.providers as xyz #para escoger el basemap
import seismicup as sup
import urllib.request
import matplotlib.font_manager as fm
import random

from matplotlib.colors import LogNorm # para la escala logaritmica de los colores
from matplotlib.ticker import LogFormatter, LogLocator # escala log
from scipy import stats # regresion lineal

# Conectar al Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiar Fuente

In [ ]:
sup.plots.get_TimesNewRoman_font()

# Paths

In [ ]:
path_save_figures = '/content/drive/MyDrive/Contratos/20250801_UIS_CO2/Notebooks/Figuras_03'

In [ ]:
path_datasets = '/content/SeismicUP/Datasets_'

os.listdir(path_datasets)

# Cargar catalogos

In [ ]:
lista_catalogos = ['Cat_sis_SGC_jun1993_feb2018',
                   'Cat_sis_SGC_mar2018_2024',
                   'Cat_sis_SGC_TECTO_feb2014_2024',
                   'Cat_sis_SGC_LBG_jun1993_2021']

for i in lista_catalogos:
  cantidad_archivos = len(os.listdir(os.path.join(path_datasets, i)))
  print(f'{i} = ', cantidad_archivos)

In [ ]:
df_general = pd.DataFrame(columns=['Fecha-Hora UTC',
                                   'Latitud', # grados
                                   'Longitud', # grados
                                   'Profundidad [km]',
                                   'Magnitud',
                                   'Tipo Magnitud',
                                   'Error Latitud [km]',
                                   'Error Longitud [km]',
                                   'Error Profundidad [km]',
                                   'Numero de Fases',
                                   'RMS [seg]',
                                   'Gap', # grados
                                   'Departamento',
                                   'Municipio',
                                   ])

df_general.head()

## Catalogo SGC 1

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_jun1993_feb2018')
os.listdir(path_catalogo)

df_SGC_1 = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_1(path_catalogo=path_catalogo,
                                            file_excel=i)

    df_SGC_1 = pd.concat([df_SGC_1, df_temp_1], ignore_index=True)
    df_SGC_1.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_1['Catalogo'] = 'SGC 1'
df_SGC_1['Gap'] = df_SGC_1['Gap'].astype(float)
df_SGC_1.info()

In [ ]:
df_SGC_1.head()

## Catalogo SGC 2

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_mar2018_2024')
os.listdir(path_catalogo)

df_SGC_2 = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_2(path_catalogo=path_catalogo,
                                            file_excel=i)

    df_SGC_2 = pd.concat([df_SGC_2, df_temp_1], ignore_index=True)
    df_SGC_2.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_2['Catalogo'] = 'SGC 2'
df_SGC_2['Gap'] = df_SGC_2['Gap'].astype(float)
df_SGC_2.info()

In [ ]:
df_SGC_2.head()

## Catalogo TECTO

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_TECTO_feb2014_2024')
os.listdir(path_catalogo)

df_SGC_TECTO = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx') and (i != 'excel_estaciones.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_TECTO(path_catalogo=path_catalogo,
                                                file_excel=i)

    print(df_temp_1.shape)
    df_SGC_TECTO = pd.concat([df_SGC_TECTO, df_temp_1], ignore_index=True)
    df_SGC_TECTO.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_TECTO['Catalogo'] = 'TECTO'
df_SGC_TECTO['Profundidad [km]'] = df_SGC_TECTO['Profundidad [km]'].astype(float)
df_SGC_TECTO['Gap'] = df_SGC_TECTO['Gap'].astype(float)
df_SGC_TECTO.info()

In [ ]:
df_SGC_TECTO.head()

## Catalogo LBG

In [ ]:
path_catalogo = os.path.join(path_datasets, 'Cat_sis_SGC_LBG_jun1993_2021')
os.listdir(path_catalogo)

df_SGC_LBG = df_general.copy()

for i in os.listdir(path_catalogo)[:]:
  if i.endswith('.xlsx'):
    print('-----'*3)
    print('File: ', i)
    print('-----'*3)
    df_temp_1 = sup.io.crear_catalogo_SGC_LBG(path_catalogo=path_catalogo,
                                              file_excel=i)

    print(df_temp_1.shape)
    df_SGC_LBG = pd.concat([df_SGC_LBG, df_temp_1], ignore_index=True)
    df_SGC_LBG.reset_index(drop=True, inplace=True)

    del df_temp_1


In [ ]:
df_SGC_LBG['Catalogo'] = 'LBG'
df_SGC_LBG.info()

In [ ]:
df_SGC_LBG.head()

# Unir catalogos

In [ ]:
df_unido = pd.concat([df_SGC_1, df_SGC_2, df_SGC_TECTO, df_SGC_LBG], ignore_index=True)
df_unido.reset_index(drop=True, inplace=True)

df_unido['Tiempo'] = pd.to_datetime(df_unido['Fecha-Hora UTC'])

df_unido.head()

In [ ]:
df_unido['Catalogo'].unique()

In [ ]:
df_unido.info()

In [ ]:
df_unido.describe().round(2)

## Duplicados

In [ ]:
df_unido.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
#  duplicados

In [ ]:
df_unido['Tipo Magnitud'].unique()

In [ ]:
df_unido['Tipo Magnitud 2'] = df_unido['Tipo Magnitud'].str.lower()
df_unido['Tipo Magnitud 2'].unique()

In [ ]:
df_unido.describe()

In [ ]:
df_unido.sort_values(by=['Tiempo', 'Tipo Magnitud 2'],
                     ascending=True,
                     inplace=True)

Se considerarán únicamente los eventos que cuenten con ML como tipo de magnitud.

In [ ]:
filtro_ml = df_unido['Tipo Magnitud 2'].str.contains('ml')

df_unido[filtro_ml]['Tipo Magnitud 2'].unique()

In [ ]:
df_unido.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# 22154 duplicados

In [ ]:
df_filtrado = df_unido.copy()
df_filtrado.drop_duplicates(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first', inplace=True)
df_filtrado.reset_index(drop=True, inplace=True)

In [ ]:
df_filtrado.duplicated(subset=['Fecha-Hora UTC', 'Latitud', 'Longitud'], keep='first').sum()
# Ya no se tienen duplicados

In [ ]:
df_filtrado.describe().round(2)

## Geodataframe

In [ ]:
gdf_filtrado = gpd.GeoDataFrame(
    df_filtrado, geometry=gpd.points_from_xy(df_filtrado.Longitud, df_filtrado.Latitud), crs="EPSG:4326"
)

In [ ]:
gdf_filtrado = gdf_filtrado.to_crs(9377)

In [ ]:
df_filtrado['X [m]'] = gdf_filtrado.get_coordinates()['x']
df_filtrado['Y [m]'] = gdf_filtrado.get_coordinates()['y']

df_filtrado['X [km]'] = df_filtrado['X [m]'] / 1000
df_filtrado['Y [km]'] = df_filtrado['Y [m]'] / 1000

df_filtrado.head()

## Punto referencia Campo Colorado



In [ ]:
punto_xy_CampoColorado = (4919222.973, 2308553.476) # x, y en metros

# se va calcular la distancia de los eventos al centroide o punto de referencia
# del Campo Colorado solo en XY mas no en profundidad

df_filtrado['Distancia [m]'] = np.sqrt((df_filtrado['X [m]'] - punto_xy_CampoColorado[0])**2 + (df_filtrado['Y [m]'] - punto_xy_CampoColorado[1])**2)
df_filtrado['Distancia [km]'] = df_filtrado['Distancia [m]'] / 1000

df_filtrado.describe().round(2)

## Subcatalogos

In [ ]:
limites_BSN = {'longitud': [-73.40, -72.80],
               'latitud': [6.50, 7.10]}
limites_CC = {'longitud': [-74.19, -73.26],
              'latitud': [6.32, 7.25]}


In [ ]:
df_FC = df_filtrado.copy()

filtro_BSN = (df_filtrado['Longitud'] >= limites_BSN['longitud'][0]) & (df_filtrado['Longitud'] <= limites_BSN['longitud'][1]) & (df_filtrado['Latitud'] >= limites_BSN['latitud'][0]) & (df_filtrado['Latitud'] <= limites_BSN['latitud'][1])
df_BSN = df_filtrado[filtro_BSN].copy()

filtro_CC = (df_filtrado['Longitud'] >= limites_CC['longitud'][0]) & (df_filtrado['Longitud'] <= limites_CC['longitud'][1]) & (df_filtrado['Latitud'] >= limites_CC['latitud'][0]) & (df_filtrado['Latitud'] <= limites_CC['latitud'][1])
df_CC = df_filtrado[filtro_CC].copy()


In [ ]:
df_FC.describe().round(2)

In [ ]:
df_BSN.describe().round(2)

In [ ]:
df_CC.describe().round(2)

In [ ]:
total_eventos_FC = df_FC.shape[0]
total_eventos_BSN = df_BSN.shape[0]
total_eventos_CC = df_CC.shape[0]

porcentaje_eventos_FC = (total_eventos_FC / total_eventos_FC) * 100
porcentaje_eventos_BSN = (total_eventos_BSN / total_eventos_FC) * 100
porcentaje_eventos_CC = (total_eventos_CC / total_eventos_FC) * 100

print(f'Total de eventos FC: {total_eventos_FC} [{porcentaje_eventos_FC:.2f}%]')
print(f'Total de eventos BSN: {total_eventos_BSN} [{porcentaje_eventos_BSN:.2f}%]')
print(f'Total de eventos CC: {total_eventos_CC} [{porcentaje_eventos_CC:.2f}%]')

Después de aplicar el filtrado del catálogo (considerando únicamente eventos con ML como tipo de magnitud), se obtiene:

* `FC` (Full Catalog): 281086 eventos (100% del conjunto de estudio).
* `BSN` (Bucaramanga Seismic Nest Catalog): 161158 eventos, equivalentes al 57.33% del `FC`.
* `CC` (Campo Colorado Catalog): 22052 eventos, equivalentes al 7.85% del `FC`.


# Histogramas errores

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(12,8))

# FC
sup.plots.histograma_plot(df= df_FC,
                          columna= 'X [km]',
                          ax_h= ax[0,0],
                          dist_min= df_FC['X [km]'].min(),
                          units_label='km',
                          step_= 20)
sup.plots.histograma_plot(df= df_FC,
                          columna= 'Error Longitud [km]',
                          ax_h= ax[1,0],
                          dist_min= 0,
                          units_label='km',
                          step_= 10)
ax[0,0].set_xlabel('X [km] - FC')
ax[1,0].set_xlabel('X Error [km] - FC')

# BSN
sup.plots.histograma_plot(df= df_BSN,
                          columna= 'X [km]',
                          ax_h= ax[0,1],
                          dist_min= df_BSN['X [km]'].min(),
                          units_label='km',
                          step_= 5)
sup.plots.histograma_plot(df= df_BSN,
                          columna= 'Error Longitud [km]',
                          ax_h= ax[1,1],
                          dist_min= 0,
                          units_label='km',
                          step_= 5)
ax[0,1].set_xlabel('X [km] - BSN')
ax[1,1].set_xlabel('X Error [km] - BSN')

# CC
sup.plots.histograma_plot(df= df_CC,
                          columna= 'X [km]',
                          ax_h= ax[0,2],
                          dist_min= df_CC['X [km]'].min(),
                          units_label='km',
                          step_= 5)
sup.plots.histograma_plot(df= df_CC,
                          columna= 'Error Longitud [km]',
                          ax_h= ax[1,2],
                          dist_min= 0,
                          units_label='km',
                          step_= 5)
ax[0,2].set_xlabel('X [km] - CC')
ax[1,2].set_xlabel('X Error [km] - CC')

ax[0,0].set_title('FC Catalog')
ax[0,1].set_title('BSN Catalog')
ax[0,2].set_title('CC Catalog')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'hits_errores_coorX',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(12,8))

# FC
sup.plots.histograma_plot(df= df_FC,
                          columna= 'Y [km]',
                          ax_h= ax[0,0],
                          dist_min= df_FC['Y [km]'].min(),
                          step_= 20)
sup.plots.histograma_plot(df= df_FC,
                          columna= 'Error Latitud [km]',
                          ax_h= ax[1,0],
                          dist_min= 0,
                          step_= 10)
ax[0,0].set_xlabel('Y [km] - FC')
ax[1,0].set_xlabel('Y Error [km] - FC')

# BSN
sup.plots.histograma_plot(df= df_BSN,
                          columna= 'Y [km]',
                          ax_h= ax[0,1],
                          dist_min= df_BSN['Y [km]'].min(),
                          step_= 5)
sup.plots.histograma_plot(df= df_BSN,
                          columna= 'Error Latitud [km]',
                          ax_h= ax[1,1],
                          dist_min= 0,
                          step_= 5)
ax[0,1].set_xlabel('Y [km] - BSN')
ax[1,1].set_xlabel('Y Error [km] - BSN')

# CC
sup.plots.histograma_plot(df= df_CC,
                          columna= 'Y [km]',
                          ax_h= ax[0,2],
                          dist_min= df_CC['Y [km]'].min(),
                          step_= 5)
sup.plots.histograma_plot(df= df_CC,
                          columna= 'Error Latitud [km]',
                          ax_h= ax[1,2],
                          dist_min= 0,
                          step_= 5)
ax[0,2].set_xlabel('Y [km] - CC')
ax[1,2].set_xlabel('Y Error [km] - CC')

ax[0,0].set_title('FC Catalog')
ax[0,1].set_title('BSN Catalog')
ax[0,2].set_title('CC Catalog')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'hits_errores_coorY',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(12,8))

# FC
sup.plots.histograma_plot(df= df_FC,
                          columna= 'Profundidad [km]',
                          ax_h= ax[0,0],
                          dist_min= df_FC['Profundidad [km]'].min(),
                          step_= 20)
sup.plots.histograma_plot(df= df_FC,
                          columna= 'Error Profundidad [km]',
                          ax_h= ax[1,0],
                          dist_min= 0,
                          step_= 20)
ax[0,0].set_xlabel('Depth [km] - FC')
ax[1,0].set_xlabel('Depth Error [km] - FC')

# BSN
sup.plots.histograma_plot(df= df_BSN,
                          columna= 'Profundidad [km]',
                          ax_h= ax[0,1],
                          dist_min= df_BSN['Profundidad [km]'].min(),
                          step_= 20)
sup.plots.histograma_plot(df= df_BSN,
                          columna= 'Error Profundidad [km]',
                          ax_h= ax[1,1],
                          dist_min= 0,
                          step_= 5)
ax[0,1].set_xlabel('Depth [km] - BSN')
ax[1,1].set_xlabel('Depth Error [km] - BSN')

# CC
sup.plots.histograma_plot(df= df_CC,
                          columna= 'Profundidad [km]',
                          ax_h= ax[0,2],
                          dist_min= df_CC['Profundidad [km]'].min(),
                          step_= 10)
sup.plots.histograma_plot(df= df_CC,
                          columna= 'Error Profundidad [km]',
                          ax_h= ax[1,2],
                          dist_min= 0,
                          step_= 5)
ax[0,2].set_xlabel('Depth [km] - CC')
ax[1,2].set_xlabel('Depth Error [km] - CC')

ax[0,0].set_title('FC Catalog')
ax[0,1].set_title('BSN Catalog')
ax[0,2].set_title('CC Catalog')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'hits_errores_profundidad',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

# Tiempo vs magnitud

## Profundidad limite

In [ ]:
profundidad_corte = 60

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=3, figsize=(8,11))

# FC
filtro_datos = df_FC['Profundidad [km]'] <= profundidad_corte
sup.plots.time_magnitud_plot(df = df_FC,
                             filtro_datos = filtro_datos,
                             columna_eje_x_tiempo = 'Tiempo',
                             comlumna_eje_y = 'Magnitud',
                             ax_s = ax[0],
                             label_s= f'{filtro_datos.sum()} Earthquakes [Depth $\leq{profundidad_corte}$ km]')
ax[0].legend()
ax[0].set_xlabel('Time')
ax[0].set_ylabel('Magnitude')
ax[0].set_title('FC Catalog')


# BSN
filtro_datos = df_BSN['Profundidad [km]'] <= profundidad_corte
sup.plots.time_magnitud_plot(df = df_BSN,
                             filtro_datos = filtro_datos,
                             columna_eje_x_tiempo = 'Tiempo',
                             comlumna_eje_y = 'Magnitud',
                             ax_s = ax[1],
                             label_s= f'{filtro_datos.sum()} Earthquakes [Depth $\leq{profundidad_corte}$ km]')
ax[1].legend()
ax[1].set_xlabel('Time')
ax[1].set_ylabel('Magnitude')
ax[1].set_title('BSN Catalog')

# CC
filtro_datos = df_CC['Profundidad [km]'] <= profundidad_corte
sup.plots.time_magnitud_plot(df = df_CC,
                             filtro_datos = filtro_datos,
                             columna_eje_x_tiempo = 'Tiempo',
                             comlumna_eje_y = 'Magnitud',
                             ax_s = ax[2],
                             label_s= f'{filtro_datos.sum()} Earthquakes [Depth $\leq{profundidad_corte}$ km]')
ax[2].legend()
ax[2].set_xlabel('Time')
ax[2].set_ylabel('Magnitude')
ax[2].set_title('CC Catalog')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'scatter_tiempo_magnitud_somero',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=3, figsize=(8,11))

# FC
filtro_datos = df_FC['Profundidad [km]'] > profundidad_corte
sup.plots.time_magnitud_plot(df = df_FC,
                             filtro_datos = filtro_datos,
                             columna_eje_x_tiempo = 'Tiempo',
                             comlumna_eje_y = 'Magnitud',
                             ax_s = ax[0],
                             label_s= f'{filtro_datos.sum()} Earthquakes [Depth $>{profundidad_corte}$ km]')
ax[0].legend()
ax[0].set_xlabel('Time')
ax[0].set_ylabel('Magnitude')
ax[0].set_title('FC Catalog')


# BSN
filtro_datos = df_BSN['Profundidad [km]'] > profundidad_corte
sup.plots.time_magnitud_plot(df = df_BSN,
                             filtro_datos = filtro_datos,
                             columna_eje_x_tiempo = 'Tiempo',
                             comlumna_eje_y = 'Magnitud',
                             ax_s = ax[1],
                             label_s= f'{filtro_datos.sum()} Earthquakes [Depth $>{profundidad_corte}$ km]')
ax[1].legend()
ax[1].set_xlabel('Time')
ax[1].set_ylabel('Magnitude')
ax[1].set_title('BSN Catalog')

# CC
filtro_datos = df_CC['Profundidad [km]'] > profundidad_corte
sup.plots.time_magnitud_plot(df = df_CC,
                             filtro_datos = filtro_datos,
                             columna_eje_x_tiempo = 'Tiempo',
                             comlumna_eje_y = 'Magnitud',
                             ax_s = ax[2],
                             label_s= f'{filtro_datos.sum()} Earthquakes [Depth $>{profundidad_corte}$ km]')
ax[2].legend()
ax[2].set_xlabel('Time')
ax[2].set_ylabel('Magnitude')
ax[2].set_title('CC Catalog')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'scatter_tiempo_magnitud_profundos',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

# Histograma magnitudes

In [ ]:
steps_magnitud = 0.2
mag_min = 0
mag_max = 8

fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(12,8))

# FC
# <= profunidad corte
filtro_datos = df_FC['Profundidad [km]'] <= profundidad_corte
df_FC_temp = df_FC[filtro_datos]
sup.plots.histograma_plot(df= df_FC_temp,
                          columna= 'Magnitud',
                          ax_h= ax[0,0],
                          dist_min= mag_min,
                          dist_max= mag_max,
                          units_label='Ml',
                          step_= steps_magnitud)
ax[0,0].set_xlabel(f'Magnitude [Depth $\leq{profundidad_corte}$ km] - FC')
# > profunidad corte
filtro_datos = df_FC['Profundidad [km]'] > profundidad_corte
df_FC_temp = df_FC[filtro_datos]
sup.plots.histograma_plot(df= df_FC_temp,
                          columna= 'Magnitud',
                          ax_h= ax[1,0],
                          dist_min= mag_min,
                          dist_max= mag_max,
                          units_label='Ml',
                          step_= steps_magnitud)
ax[1,0].set_xlabel(f'Magnitude [Depth $>{profundidad_corte}$ km] - FC')

# BSN
# <= profunidad corte
filtro_datos = df_BSN['Profundidad [km]'] <= profundidad_corte
df_BSN_temp = df_BSN[filtro_datos]
sup.plots.histograma_plot(df= df_BSN_temp,
                          columna= 'Magnitud',
                          ax_h= ax[0,1],
                          dist_min= mag_min,
                          dist_max= mag_max,
                          units_label='Ml',
                          step_= steps_magnitud)
ax[0,1].set_xlabel(f'Magnitude [Depth $\leq{profundidad_corte}$ km] - BSN')
# > profunidad corte
filtro_datos = df_BSN['Profundidad [km]'] > profundidad_corte
df_BSN_temp = df_BSN[filtro_datos]
sup.plots.histograma_plot(df= df_BSN_temp,
                          columna= 'Magnitud',
                          ax_h= ax[1,1],
                          dist_min= mag_min,
                          dist_max= mag_max,
                          units_label='Ml',
                          step_= steps_magnitud)
ax[1,1].set_xlabel(f'Magnitude [Depth $>{profundidad_corte}$ km] - BSN')

# CC
# <= profunidad corte
filtro_datos = df_CC['Profundidad [km]'] <= profundidad_corte
df_CC_temp = df_CC[filtro_datos]
sup.plots.histograma_plot(df= df_CC_temp,
                          columna= 'Magnitud',
                          ax_h= ax[0,2],
                          dist_min= mag_min,
                          dist_max= mag_max,
                          units_label='Ml',
                          step_= steps_magnitud)
ax[0,2].set_xlabel(f'Magnitude [Depth $\leq{profundidad_corte}$ km] - CC')
# > profunidad corte
filtro_datos = df_CC['Profundidad [km]'] > profundidad_corte
df_CC_temp = df_CC[filtro_datos]
sup.plots.histograma_plot(df= df_CC_temp,
                          columna= 'Magnitud',
                          ax_h= ax[1,2],
                          dist_min= mag_min,
                          dist_max= mag_max,
                          units_label='Ml',
                          step_= steps_magnitud)
ax[1,2].set_xlabel(f'Magnitude [Depth $>{profundidad_corte}$ km] - CC')

ax[0,0].set_title('FC Catalog')
ax[0,1].set_title('BSN Catalog')
ax[0,2].set_title('CC Catalog')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'hits_magnitudes_',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

## Cantidad eventos al año

In [ ]:
year_range = np.arange(df_FC['Tiempo'].dt.year.min(),
                       df_FC['Tiempo'].dt.year.max()+1)

fig, ax = plt.subplots(ncols=1, nrows=3, figsize=(7,9), sharex=True)

# Todos
sns.histplot(x = df_FC['Tiempo'].dt.year,
             #log_scale = (False, True),
             bins = year_range,
             ax= ax[0],
             legend= True,
             label= f'FC Catalog',
             element='step',
             fill=False)
sns.histplot(x = df_BSN['Tiempo'].dt.year,
             #log_scale = (False, True),
             bins = year_range,
             ax= ax[0],
             legend= True,
             label= f'BSN Catalog',
             element='step',
             fill=False)
sns.histplot(x = df_CC['Tiempo'].dt.year,
             #log_scale = (False, True),
             bins = year_range,
             ax= ax[0],
             legend= True,
             label= f'CC Catalog',
             element='step',
             fill=False)
ax[0].set_yscale('log')
ax[0].grid(ls='--', c='gray', alpha=0.7)
ax[0].legend()
ax[0].set_ylabel('Number of earthquakes')

# Someros
filtro_datos = df_FC['Profundidad [km]'] <= profundidad_corte
df_FC_temp = df_FC[filtro_datos].copy()
sns.histplot(x = df_FC_temp['Tiempo'].dt.year,
             #log_scale = (False, True),
             bins = year_range,
             ax= ax[1],
             legend= True,
             label= f'FC Catalog',
             element='step',
             fill=False)

filtro_datos = df_BSN['Profundidad [km]'] <= profundidad_corte
df_BSN_temp = df_BSN[filtro_datos].copy()
sns.histplot(x = df_BSN_temp['Tiempo'].dt.year,
             #log_scale = (False, True),
             bins = year_range,
             ax= ax[1],
             legend= True,
             label= f'BSN Catalog',
             element='step',
             fill=False)


filtro_datos = df_CC['Profundidad [km]'] > profundidad_corte
df_CC_temp = df_CC_temp[filtro_datos].copy()
sns.histplot(x = df_CC['Tiempo'].dt.year,
             #log_scale = (False, True),
             bins = year_range,
             ax= ax[1],
             legend= True,
             label= f'CC Catalog',
             element='step',
             fill=False)
ax[1].set_yscale('log')
ax[1].grid(ls='--', c='gray', alpha=0.7)
ax[1].legend()
ax[1].set_ylabel('Number of earthquakes')

# Profundos
filtro_datos = df_FC['Profundidad [km]'] > profundidad_corte
df_FC_temp = df_FC[filtro_datos].copy()
sns.histplot(x = df_FC_temp['Tiempo'].dt.year,
             #log_scale = (False, True),
             bins = year_range,
             ax= ax[2],
             legend= True,
             label= f'FC Catalog',
             element='step',
             fill=False)

filtro_datos = df_BSN['Profundidad [km]'] > profundidad_corte
df_BSN_temp = df_BSN[filtro_datos].copy()
sns.histplot(x = df_BSN_temp['Tiempo'].dt.year,
             #log_scale = (False, True),
             bins = year_range,
             ax= ax[2],
             legend= True,
             label= f'BSN Catalog',
             element='step',
             fill=False)


filtro_datos = df_CC['Profundidad [km]'] <= profundidad_corte
df_CC_temp = df_CC_temp[filtro_datos].copy()
sns.histplot(x = df_CC['Tiempo'].dt.year,
             #log_scale = (False, True),
             bins = year_range,
             ax= ax[2],
             legend= True,
             label= f'CC Catalog',
             element='step',
             fill=False)
ax[2].set_yscale('log')
ax[2].grid(ls='--', c='gray', alpha=0.7)
ax[2].legend()
ax[2].set_ylabel('Number of earthquakes')
ax[2].set_xlabel('Time')

#ax[2].set_xlabel('')
ax[2].legend()
ax[2].grid(ls='--', c='gray', alpha=0.7)


ax[0].set_title(f'All depths')
ax[1].set_title(f'Depth $\leq{profundidad_corte}$ km')
ax[2].set_title(f'Depth $>{profundidad_corte}$ km')

ax[2].set_xticks(np.arange(start=1990, stop=2025, step=2))
ax[2].set_xlim(left=1993, right=2024)

plt.suptitle(f'Number of earthquakes per year')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = f'cantidad_eventos_anuales',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

En la gráfica de número de eventos por año, el BSN muestra menos de 100 eventos anuales para profundidades someras, lo que dificulta la estimación confiable de los parámetros de Gutenberg–Richter a escala anual. Por esta razón, los cálculos se realizarán de manera acumulativa en etapas posteriores del análisis, y no de forma anual.

# Calculos N

In [ ]:
mag_min = 0
mag_max = 10
steps_mag = 0.1

## Todo el catalogo

In [ ]:
diccionario_FC = {}
diccionario_BSN = {}
diccionario_CC = {}

In [ ]:
diccionario_FC['get_N'] = sup.stats.obtener_N_LGR(magnitudes_array = df_FC['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

diccionario_FC['get_N']

In [ ]:
diccionario_BSN['get_N'] = sup.stats.obtener_N_LGR(magnitudes_array = df_BSN['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

diccionario_BSN['get_N']

In [ ]:
diccionario_CC['get_N'] = sup.stats.obtener_N_LGR(magnitudes_array = df_CC['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

diccionario_CC['get_N']

## Eventos someros

In [ ]:
diccionario_FC_somero = {}
diccionario_BSN_somero = {}
diccionario_CC_somero = {}

filtro_datos = df_FC['Profundidad [km]'] <= profundidad_corte
df_FC_temp = df_FC[filtro_datos]

filtro_datos = df_BSN['Profundidad [km]'] <= profundidad_corte
df_BSN_temp = df_BSN[filtro_datos]

filtro_datos = df_CC['Profundidad [km]'] <= profundidad_corte
df_CC_temp = df_CC[filtro_datos]

In [ ]:
diccionario_FC_somero['get_N'] = sup.stats.obtener_N_LGR(magnitudes_array = df_FC_temp['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

diccionario_FC_somero['get_N']

In [ ]:
diccionario_BSN_somero['get_N'] = sup.stats.obtener_N_LGR(magnitudes_array = df_BSN_temp['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

diccionario_BSN_somero['get_N']

In [ ]:
diccionario_CC_somero['get_N'] = sup.stats.obtener_N_LGR(magnitudes_array = df_CC_temp['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

diccionario_CC_somero['get_N']

## Eventos profundos

In [ ]:
diccionario_FC_profundo = {}
diccionario_BSN_profundo = {}
diccionario_CC_profundo = {}

filtro_datos = df_FC['Profundidad [km]'] > profundidad_corte
df_FC_temp = df_FC[filtro_datos]

filtro_datos = df_BSN['Profundidad [km]'] > profundidad_corte
df_BSN_temp = df_BSN[filtro_datos]

filtro_datos = df_CC['Profundidad [km]'] > profundidad_corte
df_CC_temp = df_CC[filtro_datos]

In [ ]:
diccionario_FC_profundo['get_N'] = sup.stats.obtener_N_LGR(magnitudes_array = df_FC_temp['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

diccionario_FC_profundo['get_N']

In [ ]:
diccionario_BSN_profundo['get_N'] = sup.stats.obtener_N_LGR(magnitudes_array = df_BSN_temp['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

diccionario_BSN_profundo['get_N']

In [ ]:
diccionario_CC_profundo['get_N'] = sup.stats.obtener_N_LGR(magnitudes_array = df_CC_temp['Magnitud'].values,
                                                  magnitud_minima = mag_min,
                                                  magnitud_maxima = mag_max,
                                                  steps_ = steps_mag,
                                                  texto = False)

diccionario_CC_profundo['get_N']

### Grafico N

In [ ]:
fig, ax = plt.subplots(nrows=3, ncols=3, figsize=(10,10),
                       sharey='row', sharex='all')

titles_ = ['FC', 'BSN', 'CC']
# Todos
for index_, diccionario in enumerate([diccionario_FC,
                                      diccionario_BSN,
                                      diccionario_CC]):
  sup.plots.GutenbergRichter_scatter_plot(N_count = diccionario['get_N']['N'],
                                          magnitudes= diccionario['get_N']['m'],
                                          ax_h= ax[0, index_],
                                          magnitud_maxima= mag_max)
  if index_ > 0:
    ax[0, index_].set_ylabel('')
  ax[0, index_].set_xlabel('')
  ax[0, index_].set_title(f'{titles_[index_]} - All depths')

# Someros
for index_, diccionario in enumerate([diccionario_FC_somero,
                                      diccionario_BSN_somero,
                                      diccionario_CC_somero]):
  sup.plots.GutenbergRichter_scatter_plot(N_count = diccionario['get_N']['N'],
                                          magnitudes= diccionario['get_N']['m'],
                                          ax_h= ax[1, index_],
                                          magnitud_maxima= mag_max)
  if index_ > 0:
    ax[1, index_].set_ylabel('')
  ax[1, index_].set_xlabel('')
  ax[1, index_].set_title(f'{titles_[index_]} - Depth $\leq{profundidad_corte}$ km')

# Profundos
for index_, diccionario in enumerate([diccionario_FC_profundo,
                                      diccionario_BSN_profundo,
                                      diccionario_CC_profundo]):
  sup.plots.GutenbergRichter_scatter_plot(N_count = diccionario['get_N']['N'],
                                          magnitudes= diccionario['get_N']['m'],
                                          ax_h= ax[2, index_],
                                          magnitud_maxima= mag_max)
  if index_ > 0:
    ax[2, index_].set_ylabel('')
  ax[2, index_].set_xlabel('')
  ax[2, index_].set_title(f'{titles_[index_]} - Depth $>{profundidad_corte}$ km')

ax[2, 0].set_xlabel('Magnitude')
ax[2, 1].set_xlabel('Magnitude')
ax[2, 2].set_xlabel('Magnitude')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'grafico_N_m_',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

# Calculo Gutenberg Richter

Para el cálculo de los parámetros a-value y b-value se empleará la metodología Boosted 2 (donde Mc se determina mediante el criterio de máxima curvatura), dado que en las pruebas preliminares fue la que produjo resultados más cercanos a los reportados por Londoño et al. (2019). A diferencia de dicho estudio, donde se aplican filtros que eliminan una fracción importante del catálogo, en este trabajo no se han descartado eventos, salvo el criterio de disponibilidad de ML como tipo de magnitud. Esta decisión busca conservar la muestra estadística lo más completa posible en cada subcatálogo.

Además, considerando que una parte del proyecto está orientada a la prevención sísmica, se optó por incluir la mayor cantidad de eventos registrados en la zona. Aunque la calidad de algunos eventos puede estar condicionada por diversos factores (por ejemplo, la densidad y cobertura de la red de estaciones), su inclusión aporta información sobre la ocurrencia sísmica y permite evaluar el comportamiento del catálogo bajo un enfoque más conservador.

Bajo estas consideraciones, se implementará el método Boosted utilizando un esquema de remuestreo del 80% de los datos sin reemplazo.

In [ ]:
mag_min = 0
mag_max = 10
steps_mag = 0.1
porcentaje_remuestreo = 0.8
numero_remuestreos = 1000
reemplazo = False
correcion_mc = 0.0

## Todo el catalogo

In [ ]:
diccionario_FC['Boosted'] = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_FC['Magnitud'],
                                                           percentage_data = porcentaje_remuestreo, # 0.8
                                                           magnitud_minima = mag_min,
                                                           magnitud_maxima = mag_max,
                                                           steps_ = steps_mag,
                                                           n_resamples = numero_remuestreos,
                                                           replace_ = reemplazo,
                                                           correcion_m = correcion_mc)

diccionario_FC['Boosted']

In [ ]:
diccionario_BSN['Boosted'] = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_BSN['Magnitud'],
                                                           percentage_data = porcentaje_remuestreo, # 0.8
                                                           magnitud_minima = mag_min,
                                                           magnitud_maxima = mag_max,
                                                           steps_ = steps_mag,
                                                           n_resamples = numero_remuestreos,
                                                           replace_ = reemplazo,
                                                           correcion_m = correcion_mc)

diccionario_BSN['Boosted']

In [ ]:
diccionario_CC['Boosted'] = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_CC['Magnitud'],
                                                           percentage_data = porcentaje_remuestreo, # 0.8
                                                           magnitud_minima = mag_min,
                                                           magnitud_maxima = mag_max,
                                                           steps_ = steps_mag,
                                                           n_resamples = numero_remuestreos,
                                                           replace_ = reemplazo,
                                                           correcion_m = correcion_mc)

diccionario_CC['Boosted']

## Eventos someros

In [ ]:
filtro_datos = df_FC['Profundidad [km]'] <= profundidad_corte
df_FC_temp = df_FC[filtro_datos].copy()

filtro_datos = df_BSN['Profundidad [km]'] <= profundidad_corte
df_BSN_temp = df_BSN[filtro_datos].copy()

filtro_datos = df_CC['Profundidad [km]'] <= profundidad_corte
df_CC_temp = df_CC[filtro_datos].copy()

In [ ]:
diccionario_FC_somero['Boosted'] = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_FC_temp['Magnitud'],
                                                           percentage_data = porcentaje_remuestreo, # 0.8
                                                           magnitud_minima = mag_min,
                                                           magnitud_maxima = mag_max,
                                                           steps_ = steps_mag,
                                                           n_resamples = numero_remuestreos,
                                                           replace_ = reemplazo,
                                                           correcion_m = correcion_mc)

diccionario_FC_somero['Boosted']

In [ ]:
diccionario_BSN_somero['Boosted'] = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_BSN_temp['Magnitud'],
                                                           percentage_data = porcentaje_remuestreo, # 0.8
                                                           magnitud_minima = mag_min,
                                                           magnitud_maxima = mag_max,
                                                           steps_ = steps_mag,
                                                           n_resamples = numero_remuestreos,
                                                           replace_ = reemplazo,
                                                           correcion_m = correcion_mc)

diccionario_BSN_somero['Boosted']

In [ ]:
diccionario_CC_somero['Boosted'] = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_CC_temp['Magnitud'],
                                                           percentage_data = porcentaje_remuestreo, # 0.8
                                                           magnitud_minima = mag_min,
                                                           magnitud_maxima = mag_max,
                                                           steps_ = steps_mag,
                                                           n_resamples = numero_remuestreos,
                                                           replace_ = reemplazo,
                                                           correcion_m = correcion_mc)

diccionario_CC_somero['Boosted']

## Eventos profundos

In [ ]:
filtro_datos = df_FC['Profundidad [km]'] > profundidad_corte
df_FC_temp = df_FC[filtro_datos].copy()

filtro_datos = df_BSN['Profundidad [km]'] > profundidad_corte
df_BSN_temp = df_BSN[filtro_datos].copy()

filtro_datos = df_CC['Profundidad [km]'] > profundidad_corte
df_CC_temp = df_CC[filtro_datos].copy()

In [ ]:
diccionario_FC_profundo['Boosted'] = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_FC_temp['Magnitud'],
                                                           percentage_data = porcentaje_remuestreo, # 0.8
                                                           magnitud_minima = mag_min,
                                                           magnitud_maxima = mag_max,
                                                           steps_ = steps_mag,
                                                           n_resamples = numero_remuestreos,
                                                           replace_ = reemplazo,
                                                           correcion_m = correcion_mc)

diccionario_FC_profundo['Boosted']

In [ ]:
diccionario_BSN_profundo['Boosted'] = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_BSN_temp['Magnitud'],
                                                           percentage_data = porcentaje_remuestreo, # 0.8
                                                           magnitud_minima = mag_min,
                                                           magnitud_maxima = mag_max,
                                                           steps_ = steps_mag,
                                                           n_resamples = numero_remuestreos,
                                                           replace_ = reemplazo,
                                                           correcion_m = correcion_mc)

diccionario_BSN_somero['Boosted']

In [ ]:
diccionario_CC_profundo['Boosted'] = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = df_CC_temp['Magnitud'],
                                                           percentage_data = porcentaje_remuestreo, # 0.8
                                                           magnitud_minima = mag_min,
                                                           magnitud_maxima = mag_max,
                                                           steps_ = steps_mag,
                                                           n_resamples = numero_remuestreos,
                                                           replace_ = reemplazo,
                                                           correcion_m = correcion_mc)

diccionario_CC_profundo['Boosted']

## Dataframe GR

In [ ]:
estadisticos = ['mc_mean', 'mc_std',
                'b_mean', 'b_std',
                'a_mean', 'a_std']

lista_diccionarios = {'FC': diccionario_FC,
                      'BSN': diccionario_BSN,
                      'CC': diccionario_CC,
                      'FC somero': diccionario_FC_somero,
                      'BSN somero': diccionario_BSN_somero,
                      'CC somero': diccionario_CC_somero,
                      'FC profundo': diccionario_FC_profundo,
                      'BSN profundo': diccionario_BSN_profundo,
                      'CC profundo': diccionario_CC_profundo,}

diccionario_vacio = {}

for key, diccionario in lista_diccionarios.items():
  diccionario_vacio[key] = []
  for estadistico in estadisticos:
    diccionario_vacio[key].append(diccionario['Boosted'][estadistico])

df_GR_todos = pd.DataFrame(diccionario_vacio)
df_GR_todos.index = estadisticos

df_GR_todos.round(2)

In [ ]:
df_GR_todos.to_csv(path_or_buf = os.path.join(path_save_figures, 'df_GR_todos.csv'),
                   sep=';',
                   decimal=',',
                   index=True)

### Grafico GR

In [ ]:
min_max_GR = [1.0, 7.0]

fig, ax = plt.subplots(nrows=3, ncols=3, figsize=(10,10),
                       sharey='row', sharex='all')

titles_ = ['FC', 'BSN', 'CC']
# Todos
for index_, diccionario in enumerate([diccionario_FC,
                                      diccionario_BSN,
                                      diccionario_CC]):
  sup.plots.GutenbergRichter_scatter_plot(N_count = diccionario['get_N']['N'],
                                          magnitudes = diccionario['get_N']['m'],
                                          ax_h = ax[0, index_],
                                          mc = diccionario['Boosted']['mc_mean'].round(2),
                                          magnitud_maxima= mag_max)
  sup.plots.GutenbergRichter_line_plot(b_value = diccionario['Boosted']['b_mean'],
                                       a_value = diccionario['Boosted']['a_mean'],
                                       ax_h = ax[0, index_],
                                       magnitud_minima = min_max_GR[0],
                                       magnitud_maxima = min_max_GR[1])
  if index_ > 0:
    ax[0, index_].set_ylabel('')
  ax[0, index_].set_xlabel('')
  ax[0, index_].set_title(f'{titles_[index_]} - All depths')

# Someros
for index_, diccionario in enumerate([diccionario_FC_somero,
                                      diccionario_BSN_somero,
                                      diccionario_CC_somero]):
  sup.plots.GutenbergRichter_scatter_plot(N_count = diccionario['get_N']['N'],
                                          magnitudes = diccionario['get_N']['m'],
                                          ax_h = ax[1, index_],
                                          mc = diccionario['Boosted']['mc_mean'].round(2),
                                          magnitud_maxima= mag_max)
  sup.plots.GutenbergRichter_line_plot(b_value = diccionario['Boosted']['b_mean'],
                                       a_value = diccionario['Boosted']['a_mean'],
                                       ax_h = ax[1, index_],
                                       magnitud_minima = min_max_GR[0],
                                       magnitud_maxima = min_max_GR[1])
  if index_ > 0:
    ax[1, index_].set_ylabel('')
  ax[1, index_].set_xlabel('')
  ax[1, index_].set_title(f'{titles_[index_]} - Depth $\leq{profundidad_corte}$ km')

# Profundos
for index_, diccionario in enumerate([diccionario_FC_profundo,
                                      diccionario_BSN_profundo,
                                      diccionario_CC_profundo]):
  sup.plots.GutenbergRichter_scatter_plot(N_count = diccionario['get_N']['N'],
                                          magnitudes = diccionario['get_N']['m'],
                                          ax_h = ax[2, index_],
                                          mc = diccionario['Boosted']['mc_mean'].round(2),
                                          magnitud_maxima= mag_max)
  sup.plots.GutenbergRichter_line_plot(b_value = diccionario['Boosted']['b_mean'],
                                       a_value = diccionario['Boosted']['a_mean'],
                                       ax_h = ax[2, index_],
                                       magnitud_minima = min_max_GR[0],
                                       magnitud_maxima = min_max_GR[1])
  if index_ > 0:
    ax[2, index_].set_ylabel('')
  ax[2, index_].set_xlabel('')
  ax[2, index_].set_title(f'{titles_[index_]} - Depth $>{profundidad_corte}$ km')

ax[2, 0].set_xlabel('Magnitude')
ax[2, 1].set_xlabel('Magnitude')
ax[2, 2].set_xlabel('Magnitude')

plt.tight_layout()

sup.plots.save_plot(path_save_figuras = path_save_figures,
                    file_name = 'grafico_GR_boosted_',
                    formato = 'png',
                    dpi_ = 300)

plt.show()

# Calculo GR acumulado

Se procederá a calcular los parámetros a-value, b-value y Mc de forma acumulativa. Para ello, el análisis se iniciará en el período mínimo 1993–1994 y, posteriormente, se irá incorporando información de manera progresiva, añadiendo un año a la vez, hasta completar el intervalo 1993–2024.

In [ ]:
print('Año mínimo: ', df_FC['Tiempo'].dt.year.min())
print('Año máximo: ', df_FC['Tiempo'].dt.year.max())

In [ ]:
year_range = np.arange(df_FC['Tiempo'].dt.year.min(),
                       df_FC['Tiempo'].dt.year.max()+1)
year_range

## Todo el catalogo

In [ ]:
diccionario_Acumulado_todos = {}

estadisticos = ['mc_mean', 'mc_std',
                'b_mean', 'b_std',
                'a_mean', 'a_std']

catalogos_ = {'FC': df_FC,
              'BSN': df_BSN,
              'CC': df_CC}


for key, catalogo in catalogos_.items():
  print('\n')
  lista_years = []
  lista_estadisticos_label = []
  lista_estadisticos = []
  print(key)
  catalogo_temp = catalogo.copy()
  catalogo_temp.reset_index(inplace=True, drop=True)
  for i in range(1, len(year_range)):
    years_ = [year_range[0], year_range[i]]
    print(years_)
    filtro_datos = (catalogo_temp['Tiempo'].dt.year >= years_[0]) & (catalogo_temp['Tiempo'].dt.year <= years_[1])
    #filtro_datos = catalogo_temp['Tiempo'].dt.year.isin(year_range[0:i+1])
    print(year_range[0], year_range[i])
    magnitudes_tempo = catalogo_temp['Magnitud'][filtro_datos]
    print('Suma filtro', filtro_datos.sum())
    print('Shape magnitudes', magnitudes_tempo.shape)
    diccionario_temp = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = magnitudes_tempo,
                                                            percentage_data = porcentaje_remuestreo, # 0.8
                                                            magnitud_minima = mag_min,
                                                            magnitud_maxima = mag_max,
                                                            steps_ = steps_mag,
                                                            n_resamples = numero_remuestreos,
                                                            replace_ = reemplazo,
                                                            correcion_m = correcion_mc)

    for estadistico in estadisticos:
      lista_years.append(f'{year_range[0]}-{year_range[i]}')
      lista_estadisticos_label.append(estadistico)
      lista_estadisticos.append(diccionario_temp[estadistico])


  diccionario_Acumulado_todos[key] = {'Años': lista_years,
                                      'Estadistico': lista_estadisticos_label,
                                      'Valor': lista_estadisticos}

### Dataframe

In [ ]:
diccionario_Acumulado_todos.keys()

In [ ]:
diccionario_Acumulado_todos['FC'].keys()

In [ ]:
pd.DataFrame(diccionario_Acumulado_todos['FC']).head()

In [ ]:
diccionario_df_acumulados = {}

for key, diccionario in diccionario_Acumulado_todos.items():
  df_temp = pd.DataFrame(diccionario)
  estadisticos = df_temp['Estadistico'].unique()

  df_vacio = pd.DataFrame(columns = ['Años'])
  df_vacio['Años'] = df_temp['Años'].unique()
  df_vacio['Catalogo'] = key
  #df_vacio.set_index('Años', inplace=True)

  for estadistico in estadisticos:
    df_vacio[estadistico] = df_temp[df_temp['Estadistico'] == estadistico]['Valor'].values

  diccionario_df_acumulados[key] = df_vacio


In [ ]:
diccionario_df_acumulados.keys()

In [ ]:
diccionario_df_acumulados['FC']

In [ ]:
for key, df in diccionario_df_acumulados.items():
  df.to_csv(path_or_buf = os.path.join(path_save_figures, f'df_GR_acumulado_{key}_todos.csv'),
            sep=';',
            decimal=',',
            index=True)

## Eventos someros

In [ ]:
diccionario_Acumulado_someros = {}

estadisticos = ['mc_mean', 'mc_std',
                'b_mean', 'b_std',
                'a_mean', 'a_std']

catalogos_ = {'FC somero': df_FC,
              'BSN somero': df_BSN,
              'CC somero': df_CC}


for key, catalogo in catalogos_.items():
  print('\n')
  lista_years = []
  lista_estadisticos_label = []
  lista_estadisticos = []
  filtro_datos = catalogo['Profundidad [km]'] <= profundidad_corte
  catalogo_temp = catalogo[filtro_datos].copy()
  catalogo_temp.reset_index(inplace=True, drop=True)
  print(key)
  for i in range(1, len(year_range)):
    years_ = [year_range[0], year_range[i]]
    print(years_)
    filtro_datos = (catalogo_temp['Tiempo'].dt.year >= years_[0]) & (catalogo_temp['Tiempo'].dt.year <= years_[1])
    #filtro_datos = catalogo_temp['Tiempo'].dt.year.isin([year_range[0], year_range[i]])
    print(year_range[0], year_range[i])
    magnitudes_tempo = catalogo_temp['Magnitud'][filtro_datos]
    print('Suma filtro', filtro_datos.sum())
    print('Shape magnitudes', magnitudes_tempo.shape)
    diccionario_temp = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = magnitudes_tempo,
                                                            percentage_data = porcentaje_remuestreo, # 0.8
                                                            magnitud_minima = mag_min,
                                                            magnitud_maxima = mag_max,
                                                            steps_ = steps_mag,
                                                            n_resamples = numero_remuestreos,
                                                            replace_ = reemplazo,
                                                            correcion_m = correcion_mc)

    for estadistico in estadisticos:
      lista_years.append(f'{year_range[0]}-{year_range[i]}')
      lista_estadisticos_label.append(estadistico)
      lista_estadisticos.append(diccionario_temp[estadistico])

  diccionario_Acumulado_someros[key] = {'Años': lista_years,
                                      'Estadistico': lista_estadisticos_label,
                                      'Valor': lista_estadisticos}

### Dataframe

In [ ]:
diccionario_Acumulado_someros.keys()

In [ ]:
diccionario_Acumulado_someros['FC somero'].keys()

In [ ]:
pd.DataFrame(diccionario_Acumulado_someros['FC somero']).head()

In [ ]:
diccionario_df_acumulados_somero = {}

for key, diccionario in diccionario_Acumulado_someros.items():
  df_temp = pd.DataFrame(diccionario)
  estadisticos = df_temp['Estadistico'].unique()

  df_vacio = pd.DataFrame(columns = ['Años'])
  df_vacio['Años'] = df_temp['Años'].unique()
  df_vacio['Catalogo'] = key
  #df_vacio.set_index('Años', inplace=True)

  for estadistico in estadisticos:
    df_vacio[estadistico] = df_temp[df_temp['Estadistico'] == estadistico]['Valor'].values

  diccionario_df_acumulados_somero[key] = df_vacio

In [ ]:
diccionario_df_acumulados_somero.keys()

In [ ]:
diccionario_df_acumulados_somero['FC somero']

In [ ]:
for key, df in diccionario_df_acumulados_somero.items():
  df.to_csv(path_or_buf = os.path.join(path_save_figures, f'df_GR_acumulado_{key}_.csv'),
            sep=';',
            decimal=',',
            index=True)

## Eventos profundos

In [ ]:
diccionario_Acumulado_profundo = {}

estadisticos = ['mc_mean', 'mc_std',
                'b_mean', 'b_std',
                'a_mean', 'a_std']

catalogos_ = {'FC profundo': df_FC,
              'BSN profundo': df_BSN,
              'CC profundo': df_CC}


for key, catalogo in catalogos_.items():
  print('\n')
  lista_years = []
  lista_estadisticos_label = []
  lista_estadisticos = []
  filtro_datos = catalogo['Profundidad [km]'] > profundidad_corte
  catalogo_temp = catalogo[filtro_datos].copy()
  catalogo_temp.reset_index(inplace=True, drop=True)
  print(key)
  for i in range(1, len(year_range)):
    years_ = [year_range[0], year_range[i]]
    print(years_)
    filtro_datos = (catalogo_temp['Tiempo'].dt.year >= years_[0]) & (catalogo_temp['Tiempo'].dt.year <= years_[1])
    #filtro_datos = catalogo_temp['Tiempo'].dt.year.isin([year_range[0], year_range[i]])
    print(year_range[0], year_range[i])
    magnitudes_tempo = catalogo_temp['Magnitud'][filtro_datos]
    print('Suma filtro', filtro_datos.sum())
    print('Shape magnitudes', magnitudes_tempo.shape)
    diccionario_temp = sup.stats.get_GutenbergRichter_values_boosted_2(magnitudes = magnitudes_tempo,
                                                            percentage_data = porcentaje_remuestreo, # 0.8
                                                            magnitud_minima = mag_min,
                                                            magnitud_maxima = mag_max,
                                                            steps_ = steps_mag,
                                                            n_resamples = numero_remuestreos,
                                                            replace_ = reemplazo,
                                                            correcion_m = correcion_mc)

    for estadistico in estadisticos:
      lista_years.append(f'{year_range[0]}-{year_range[i]}')
      lista_estadisticos_label.append(estadistico)
      lista_estadisticos.append(diccionario_temp[estadistico])

  diccionario_Acumulado_profundo[key] = {'Años': lista_years,
                                      'Estadistico': lista_estadisticos_label,
                                      'Valor': lista_estadisticos}

### Dataframe

In [ ]:
diccionario_Acumulado_profundo.keys()

In [ ]:
diccionario_Acumulado_profundo['FC profundo'].keys()

In [ ]:
pd.DataFrame(diccionario_Acumulado_profundo['FC profundo']).head()

In [ ]:
diccionario_df_acumulados_profundo = {}

for key, diccionario in diccionario_Acumulado_profundo.items():
  df_temp = pd.DataFrame(diccionario)
  estadisticos = df_temp['Estadistico'].unique()

  df_vacio = pd.DataFrame(columns = ['Años'])
  df_vacio['Años'] = df_temp['Años'].unique()
  df_vacio['Catalogo'] = key
  #df_vacio.set_index('Años', inplace=True)

  for estadistico in estadisticos:
    df_vacio[estadistico] = df_temp[df_temp['Estadistico'] == estadistico]['Valor'].values

  diccionario_df_acumulados_profundo[key] = df_vacio

In [ ]:
diccionario_df_acumulados_profundo.keys()

In [ ]:
diccionario_df_acumulados_profundo['FC profundo']

In [ ]:
for key, df in diccionario_df_acumulados_profundo.items():
  df.to_csv(path_or_buf = os.path.join(path_save_figures, f'df_GR_acumulado_{key}_.csv'),
            sep=';',
            decimal=',',
            index=True)

## Grafico GR acumulado

In [ ]:
diccionario_df_acumulados.keys()

In [ ]:
diccionario_df_acumulados_somero.keys()

In [ ]:
diccionario_df_acumulados_profundo.keys()

In [ ]:
variables = {'b-value': ['b_mean', 'b_std'],
             'a-value': ['a_mean', 'a_std'],
             'Mc': ['mc_mean', 'mc_std']}

for var_, estadistico_ in variables.items():
  fig, ax = plt.subplots(ncols=1, nrows=3, figsize=(7,9), sharex=True)

  # Todos
  for catalogo_, dataFrame_ in diccionario_df_acumulados.items():
    ax[0].errorbar(x = year_range[1:],
                   y = dataFrame_[estadistico_[0]],
                   yerr = dataFrame_[estadistico_[1]],
                   fmt = '-o',
                   label = catalogo_[0:3])
  ax[0].set_xlabel('')
  ax[0].legend()
  ax[0].set_ylabel(f'{var_}')
  ax[0].grid(ls='--', c='gray', alpha=0.7)

  # Someros
  for catalogo_, dataFrame_ in diccionario_df_acumulados_somero.items():
    ax[1].errorbar(x = year_range[1:],
                   y = dataFrame_[estadistico_[0]],
                   yerr = dataFrame_[estadistico_[1]],
                   fmt = '-o',
                   label = catalogo_[0:3])
  ax[1].set_xlabel('')
  ax[1].legend()
  ax[1].set_ylabel(f'{var_}')
  ax[1].grid(ls='--', c='gray', alpha=0.7)

  # Profundos
  for catalogo_, dataFrame_ in diccionario_df_acumulados_profundo.items():
    ax[2].errorbar(x = year_range[1:],
                   y = dataFrame_[estadistico_[0]],
                   yerr = dataFrame_[estadistico_[1]],
                   fmt = '-o',
                   label = catalogo_[0:3])
  #ax[2].set_xlabel('')
  ax[2].legend()
  ax[2].set_ylabel(f'{var_}')
  ax[2].grid(ls='--', c='gray', alpha=0.7)


  ax[0].set_title(f'All depths')
  ax[1].set_title(f'Depth $\leq{profundidad_corte}$ km')
  ax[2].set_title(f'Depth $>{profundidad_corte}$ km')

  plt.tight_layout()

  sup.plots.save_plot(path_save_figuras = path_save_figures,
                      file_name = f'GR_{var_}_acumulado',
                      formato = 'png',
                      dpi_ = 300)

  plt.show()
  print('\n')

# Fin